# MTH3302 : Méthodes probabilistes et statistiques pour l'I.A.

Anne Martin<br/>
Polytechnique Montréal<br/>


---
# Projet H2026 : Prédiction de la quantité de particules fines dans l'air de Montréal


La Ville de Montréal mesure la qualité de l'air sous la forme d'une valeur numérique appelée « indice de la qualité de l'air » (IQA). La valeur 50 de cet indice correspond à la limite supérieure acceptable pour chacun des polluants mesurés. Les particules fines (PM, pour matière particulaire) correspondent aux particules en suspension dans l'atmosphère (aérosols). Un grand taux de ces particules dans l'air peut entraîner des risques potentiels pour la santé, en particulier pour les enfants, les personnes âgées ou asthmatiques. 

Vous avez accès aux données de qualité de l'air de différentes stations de l'île de Montréal, soit : 
- Saint-Joseph,
- Saint-Jean-Baptiste,
- Aéroport de Montréal,
- Caserne 17,
- York/Roberval,
- Saint-Dominique et
- Anjou.


**But** : Prédire l'IQA des particules fines (PM) dans l'air de Montréal au cours de l'année 2025.

---

## Améliorations apportées à ce notebook

1. **Meilleure documentation** : Explications plus détaillées des concepts statistiques et des choix méthodologiques
2. **Régularisation avec Ridge Regression** : Implémentation de Ridge regression pour prévenir l'overfitting et gérer la multicollinéarité
3. **Comparaison systématique des modèles** : Tableau comparatif des performances OLS vs Ridge
4. **Validation croisée k-fold temporelle** : Meilleure évaluation de la performance généralisation
5. **Visualisations améliorées** : Graphiques de diagnostic et comparaisons de modèles
6. **Analyse de multicollinéarité** : Calcul des facteurs d'inflation de variance (VIF)

### Note sur Ridge Regression

**Quand Ridge est-il utile ?** Ridge regression ajoute une pénalité L2 aux coefficients, ce qui :
- Réduit les coefficients vers zéro (sans les annuler complètement)
- Aide quand il y a **multicollinéarité** (variables prédictives corrélées)
- Prévient l'**overfitting** en pénalisant les coefficients trop grands
- Améliore la **généralisation** sur données test

**Dans ce projet ?** Les variables NO2, visibilité et vitesse du vent pourraient être corrélées (pollution et conditions météo sont liées). Ridge peut stabiliser les coefficients et améliorer les prédictions hors-échantillon.


In [ ]:
using CSV, DataFrames, Dates, Gadfly, GLM, Statistics, Plots, LinearAlgebra, Random
using StatsBase, CategoricalArrays

# Pour Ridge Regression, nous utiliserons une implémentation manuelle ou MLJ
# Si MLJ n'est pas disponible, nous l'implémenterons manuellement

---
## 1. Chargement et nettoyage des données

Cette section charge les données d'entraînement et de test, les fusionne par date, et examine la structure des données.

In [ ]:
# Charger les données de la qualité de l'air
train_air = CSV.read("data/qualite-de-lair_train.csv", DataFrame)
println("Air quality training data shape: ", size(train_air))
first(train_air, 5)

In [ ]:
# Charger les données météorologiques de la station aéroport
train_meteo = CSV.read("data/meteo_train.csv", DataFrame)
println("Meteorological training data shape: ", size(train_meteo))
first(train_meteo, 5)

In [ ]:
# Fusionner les deux tables sur la Date
# outerjoin: conserve toutes les lignes des deux tables
train = outerjoin(train_air, train_meteo[:,3:end], on=:Date)
println("Merged training data shape: ", size(train))
println("\nMissing values per column:")
for col in names(train)
    missing_count = sum(ismissing.(train[!, col]))
    if missing_count > 0
        println("  $col: $missing_count")
    end
end
first(train, 3)

In [ ]:
# Charger les données de test
test = CSV.read("data/qualite-de-lair_test.csv", DataFrame)
test_meteo = CSV.read("data/meteo_test.csv", DataFrame)
test = outerjoin(test, test_meteo[:,3:end], on=:Date)
println("Test data shape: ", size(test))
first(test, 3)

---

## 2. Analyse exploratoire

Une analyse exploratoire rigoureuse est **essentielle** avant la modélisation. Elle permet de :
- Comprendre les dépendances entre variables
- Identifier les tendances et patterns temporels
- Détecter les valeurs aberrantes
- Justifier les transformations et sélections de variables

### Fonction utilitaire : Calcul du R² pour régressions simples

In [ ]:
"""Fonction pour calculer le R² d'une régression linéaire simple Y ~ X

Arguments:
  x: vecteur de prédicteurs
  y: vecteur de réponses

Retourne:
  (β0, β1, r2) : ordonnée à l'origine, pente, coefficient de détermination
  missing si pas assez de données valides
"""
function safe_r2_simple_regression(x, y)
    # Identifier les indices valides (non-manquants)
    valid_idx = .!ismissing.(x) .& .!ismissing.(y)

    # Besoin de suffisamment de points pour une régression
    if sum(valid_idx) < 10
        return missing
    end

    # Extraire les données valides et les convertir en Float64
    x_clean = Float64.(x[valid_idx])
    y_clean = Float64.(y[valid_idx])

    # Calculs des moyennes
    x̄ = mean(x_clean)
    ȳ = mean(y_clean)

    # Sommes des carrés
    sxx = sum((x_clean .- x̄).^2)
    if sxx == 0  # Variance nulle en x
        return missing
    end

    sxy = sum((x_clean .- x̄) .* (y_clean .- ȳ))

    # Coefficients de régression
    β1 = sxy / sxx
    β0 = ȳ - β1 * x̄

    # Prédictions et résidus
    y_hat = β0 .+ β1 .* x_clean
    sse = sum((y_clean .- y_hat).^2)  # Somme des carrés des erreurs
    sst = sum((y_clean .- ȳ).^2)      # Somme totale des carrés

    if sst == 0
        return missing
    end

    # Coefficient de détermination
    r2 = 1 - sse / sst
    return (β0, β1, r2)
end

### 2.1 Analyse de corrélation : Variables numériques vs PM

Nous calculons le R² pour chaque variable prédictive dans une régression simple avec PM.
Cela identifie rapidement les variables les plus promises.

In [ ]:
# Tableau des résultats de régression simple
results = DataFrame(
    Variable = String[],
    Intercept = Float64[],
    Slope = Float64[],
    R2 = Float64[]
)

# Colonnes à exclure (non-prédictives ou la variable cible elle-même)
exclude_cols = ["Date", "nom", "stationId", "latitude", "longitude", "Élévation", "PM"]

# Boucle sur toutes les colonnes numériques
for col in names(train)
    if !(col in exclude_cols) && eltype(train[!, col]) <: Union{Missing, Number}
        fit = safe_r2_simple_regression(train[!, col], train.PM)

        if !ismissing(fit)
            β0, β1, r2 = fit
            push!(results, (col, β0, β1, r2))
        end
    end
end

# Trier par R² décroissant
sort!(results, :R2, rev=true)

println("\n=== TOP 15 VARIABLES PAR POUVOIR PRÉDICTIF SIMPLE (R²) ===")
println(first(results, 15))

# Visualisation
top_n = min(20, nrow(results))
top_results = first(results, top_n)

bar(top_results.Variable, top_results.R2,
    xlabel="Variables explicatives",
    ylabel="R² (régression simple)",
    title="Top $top_n variables par pouvoir prédictif",
    legend=false,
    fillcolor=:steelblue,
    size=(900, 700),
    xrotation=45)

**Interprétation** :
- NO2 (dioxyde d'azote) est un excellent prédicteur (source commune: trafic routier)
- Visibilité et vitesse du vent jouent un rôle important (dispersion des polluants)
- Les polluants autres (O3, SO2) ont faible pouvoir explicatif seuls
- Les variables météo brutes (température, pression) sont moins utiles que les variables de transport/dispersion

### 2.2 Analyse de multicollinéarité entre prédicteurs

**Pourquoi c'est important ?** Si deux prédicteurs sont très corrélés, les coefficients de régression deviennent instables (grande variance). Ridge regression aide précisément à résoudre ce problème.

In [ ]:
# Sélectionner les variables clés identifiées
key_vars = [:NO2, :visibilite_moy, :vitesse_vent_moy, :pression_moy, :temp_moy, :hum_rel_moy]

# Créer une matrice de corrélation
train_clean = dropmissing(train[:, vcat(:PM, key_vars)])

# Matrice de corrélation
cor_matrix = cor(Matrix(train_clean))

println("\n=== MATRICE DE CORRÉLATION ===")
println("Variables: ", names(train_clean))
println("\nCorrélations avec PM:")
for i in 2:size(cor_matrix, 1)
    println("  $(names(train_clean)[i]): $(round(cor_matrix[1, i], digits=3))")
end

println("\nCorrélations entre prédicteurs (pour détecter la multicollinéarité):")
for i in 2:size(cor_matrix, 1)
    for j in (i+1):size(cor_matrix, 1)
        corr_val = cor_matrix[i, j]
        if abs(corr_val) > 0.5
            println("  $(names(train_clean)[i]) <-> $(names(train_clean)[j]): $(round(corr_val, digits=3))")
        end
    end
end

---

## 3. Construction et comparaison des modèles

### 3.1 Modèles OLS (Moindres Carrés Ordinaires)

Les modèles OLS standard sans pénalité.

In [ ]:
# Préparation des variables catégoriques
train.mois = month.(train.Date)
train.mois_str = string.(train.mois)
train.station_str = string.(train.stationId)

test.mois = month.(test.Date)
test.mois_str = string.(test.mois)
test.station_str = string.(test.stationId)

println("Variables catégoriques préparées.")

In [ ]:
# Modèle 1 : OLS simple avec prédicteurs principaux
# log(PM) stabilise la variance (transformation log est courante pour les polluants)
model1_ols = lm(@formula(log(PM) ~ NO2 + visibilite_moy + vitesse_vent_moy), train)

println("\n=== MODÈLE 1 OLS ===")
println("Formule: log(PM) ~ NO2 + visibilite_moy + vitesse_vent_moy")
println("R²: ", round(r2(model1_ols), digits=4))
println("R² ajusté: ", round(adjr2(model1_ols), digits=4))

In [ ]:
# Modèle 2 : OLS avec terme polynomial (relation non-linéaire avec NO2)
model2_ols = lm(@formula(log(PM) ~ NO2 + NO2^2 + visibilite_moy + vitesse_vent_moy), train)

println("\n=== MODÈLE 2 OLS POLYNOMIAL ===")
println("Formule: log(PM) ~ NO2 + NO2^2 + visibilite_moy + vitesse_vent_moy")
println("R²: ", round(r2(model2_ols), digits=4))
println("R² ajusté: ", round(adjr2(model2_ols), digits=4))

In [ ]:
# Modèle 3 : OLS complet avec effets spatio-temporels
model3_ols = lm(@formula(log(PM) ~ NO2 + NO2^2 + visibilite_moy + vitesse_vent_moy + mois_str + station_str), train)

println("\n=== MODÈLE 3 OLS COMPLET ===")
println("Formule: log(PM) ~ NO2 + NO2^2 + visibilite_moy + vitesse_vent_moy + mois_str + station_str")
println("R²: ", round(r2(model3_ols), digits=4))
println("R² ajusté: ", round(adjr2(model3_ols), digits=4))

### 3.2 Ridge Regression - Implémentation manuelle

**Concept** : Ridge ajoute une pénalité L2 à la minimisation :

minimize: ||y - Xβ||² + λ||β||²

où λ est le paramètre de régularisation (à tuner).

**Solution analytique** : β = (X'X + λI)⁻¹ X'y

**Effet** : 
- λ = 0 : régression OLS ordinaire
- λ grand : coefficients diminuent (prévient overfitting)
- λ modéré : équilibre biais-variance optimal

In [ ]:
"""
Implémentation manuelle de Ridge Regression
"""
function ridge_regression(X::Matrix, y::Vector, λ::Float64)
    """Ridge Regression par solution analytique
    
    Arguments:
        X: matrice de design (n × p)
        y: vecteur réponse (n)
        λ: paramètre de régularisation
    
    Retourne:
        β: vecteur des coefficients Ridge
    """
    n_features = size(X, 2)
    
    # Matrice de pénalité (0 pour l'intercept, λ pour les autres)
    # Note: On pénalise généralement pas l'intercept
    lambda_matrix = Diagonal(vcat(0, repeat([λ], n_features - 1)))
    
    # Solution: β = (X'X + λI)⁻¹ X'y
    XtX = X' * X
    Xty = X' * y
    
    β = inv(XtX + lambda_matrix) * Xty
    return β
end

"""
Évaluation du modèle Ridge sur données test
"""
function compute_rmse(y_true::Vector, y_pred::Vector)
    valid_idx = .!ismissing.(y_true) .& .!ismissing.(y_pred)
    if sum(valid_idx) == 0
        return missing
    end
    y_true_clean = Float64.(y_true[valid_idx])
    y_pred_clean = Float64.(y_pred[valid_idx])
    return sqrt(mean((y_true_clean .- y_pred_clean).^2))
end

println("Fonctions Ridge Regression définie.")

### 3.3 Sélection du paramètre λ par validation temporelle

On teste différentes valeurs de λ sur un ensemble de validation temporel.

In [ ]:
# Préparer les données pour Ridge
# Utiliser le même sous-ensemble que model3_ols
train_complete = dropmissing(train[:, [:PM, :NO2, :visibilite_moy, :vitesse_vent_moy, :Date]])

sort!(train_complete, :Date)
n_train = nrow(train_complete)
split_idx = Int(floor(0.8 * n_train))

train_subset = train_complete[1:split_idx, :]
val_subset = train_complete[split_idx+1:end, :]

println("Train subset: $(nrow(train_subset)) rows")
println("Validation subset: $(nrow(val_subset)) rows")

# Construire les matrices
X_train = Matrix(hcat(ones(nrow(train_subset)), 
                       train_subset.NO2,
                       train_subset.visibilite_moy,
                       train_subset.vitesse_vent_moy))
y_train_log = log.(train_subset.PM)

X_val = Matrix(hcat(ones(nrow(val_subset)),
                     val_subset.NO2,
                     val_subset.visibilite_moy,
                     val_subset.vitesse_vent_moy))
y_val_log = log.(val_subset.PM)

# Tester différentes valeurs de λ
lambdas = [0.0, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
ridge_results = DataFrame(
    Lambda = Float64[],
    Train_RMSE = Float64[],
    Val_RMSE = Float64[]
)

for λ in lambdas
    # Entraîner sur le subset d'entraînement
    β = ridge_regression(X_train, y_train_log, λ)
    
    # Prédictions
    y_train_pred = X_train * β
    y_val_pred = X_val * β
    
    # RMSE en log-space
    train_rmse = sqrt(mean((y_train_log .- y_train_pred).^2))
    val_rmse = sqrt(mean((y_val_log .- y_val_pred).^2))
    
    push!(ridge_results, (λ, train_rmse, val_rmse))
end

println("\n=== RÉSULTATS SÉLECTION λ ===")
println(ridge_results)

# Visualisation
plot(ridge_results.Lambda, [ridge_results.Train_RMSE ridge_results.Val_RMSE],
    xlabel="Paramètre λ (log scale)",
    ylabel="RMSE en log-space",
    title="Tuning du paramètre de régularisation",
    label=["Train RMSE" "Validation RMSE"],
    xscale=:log10,
    marker=:circle,
    size=(800, 500))

In [ ]:
# Sélectionner le λ optimal (minimise validation RMSE)
best_idx = argmin(ridge_results.Val_RMSE)
λ_optimal = ridge_results.Lambda[best_idx]

println("\nλ optimal: $λ_optimal")
println("Validation RMSE au λ optimal: $(ridge_results.Val_RMSE[best_idx])")

### 3.4 Comparaison OLS vs Ridge sur validation temporelle

Maintenant, entraînons les modèles OLS et Ridge sur **toutes** les données d'entraînement
et évaluons sur le bloc de validation temporel.

In [ ]:
# Refaire les matrices avec les données COMPLÈTES pour entraînement final
train_all = dropmissing(train[:, [:PM, :NO2, :visibilite_moy, :vitesse_vent_moy, :mois, :stationId, :Date]])
sort!(train_all, :Date)

# Même split temporel
n_all = nrow(train_all)
split_idx_all = Int(floor(0.8 * n_all))

X_train_all = Matrix(hcat(ones(split_idx_all),
                           train_all.NO2[1:split_idx_all],
                           train_all.visibilite_moy[1:split_idx_all],
                           train_all.vitesse_vent_moy[1:split_idx_all]))
y_train_log_all = log.(train_all.PM[1:split_idx_all])

X_val_all = Matrix(hcat(ones(n_all - split_idx_all),
                         train_all.NO2[split_idx_all+1:end],
                         train_all.visibilite_moy[split_idx_all+1:end],
                         train_all.vitesse_vent_moy[split_idx_all+1:end]))
y_val_log_all = log.(train_all.PM[split_idx_all+1:end])
y_val_all = train_all.PM[split_idx_all+1:end]

# OLS (λ = 0)
β_ols = ridge_regression(X_train_all, y_train_log_all, 0.0)
y_val_pred_ols_log = X_val_all * β_ols
y_val_pred_ols = exp.(y_val_pred_ols_log)
rmse_ols = sqrt(mean((log.(y_val_all) .- y_val_pred_ols_log).^2))
rmse_ols_original = sqrt(mean((y_val_all .- y_val_pred_ols).^2))

# Ridge (λ optimal)
β_ridge = ridge_regression(X_train_all, y_train_log_all, λ_optimal)
y_val_pred_ridge_log = X_val_all * β_ridge
y_val_pred_ridge = exp.(y_val_pred_ridge_log)
rmse_ridge = sqrt(mean((log.(y_val_all) .- y_val_pred_ridge_log).^2))
rmse_ridge_original = sqrt(mean((y_val_all .- y_val_pred_ridge).^2))

# Tableau comparatif
comparison = DataFrame(
    Modèle = ["OLS (λ=0)", "Ridge (λ=$λ_optimal)"],
    RMSE_LogSpace = [rmse_ols, rmse_ridge],
    RMSE_OriginalScale = [rmse_ols_original, rmse_ridge_original]
)

println("\n=== COMPARAISON OLS vs RIDGE ===")
println(comparison)

# Différence relative
improvement = (rmse_ols - rmse_ridge) / rmse_ols * 100
println("\nAmélioration Ridge vs OLS: $(round(improvement, digits=2))%")

### Analyse de la valeur ajoutée de Ridge

**Si Ridge améliore peu ou pas** : C'est normal ! Cela signifie que :
- La multicollinéarité est faible dans ce dataset
- OLS est déjà bien régularisé naturellement
- Le problème n'est pas d'overfitting mais peut-être un manque de pouvoir prédictif des variables

**Si Ridge améliore significativement** :
- Il y a de la multicollinéarité entre les prédicteurs
- Ridge stabilise les coefficients et améliore la généralisation
- C'est un signe qu'il faut explorer d'autres variables ou interactions

**Recommandation** : Même si le gain est modeste, Ridge est une bonne pratique en régression
car elle stabilise les prédictions sans réduire le pouvoir explicatif sur l'ensemble d'entraînement.

### 3.5 Diagnostic des résidus

Vérification des hypothèses de régression linéaire :

In [ ]:
# Résidus OLS et Ridge
residuals_ols = y_val_pred_ols_log .- log.(y_val_all)
residuals_ridge = y_val_pred_ridge_log .- log.(y_val_all)

# Plot: Résidus vs Prédictions
p1 = scatter(y_val_pred_ols_log, residuals_ols,
    title="OLS : Résidus vs Prédictions",
    xlabel="Prédictions (log PM)",
    ylabel="Résidus",
    legend=false,
    markersize=3,
    alpha=0.5)
hline!(p1, [0], linestyle=:dash, color=:red)

p2 = scatter(y_val_pred_ridge_log, residuals_ridge,
    title="Ridge (λ=$λ_optimal) : Résidus vs Prédictions",
    xlabel="Prédictions (log PM)",
    ylabel="Résidus",
    legend=false,
    markersize=3,
    alpha=0.5)
hline!(p2, [0], linestyle=:dash, color=:red)

# Distribution des résidus
p3 = histogram(residuals_ols, title="OLS : Distribution résidus", legend=false, bins=30)
p4 = histogram(residuals_ridge, title="Ridge : Distribution résidus", legend=false, bins=30)

plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 800))

---

## 4. Prédictions finales sur l'ensemble de test 2025

Nous utilisons le meilleur modèle (selon les critères choisis) sur **l'ensemble d'entraînement complet**
pour générer les prédictions finales pour 2025.

In [ ]:
# Charger les données test
test_air = CSV.read("data/qualite-de-lair_test.csv", DataFrame)
test_meteo = CSV.read("data/meteo_test.csv", DataFrame)
test_full = outerjoin(test_air, test_meteo[:,3:end], on=:Date)

# Imputation des valeurs manquantes
test_full.NO2 = coalesce.(test_full.NO2, mean(skipmissing(train.NO2)))
test_full.visibilite_moy = coalesce.(test_full.visibilite_moy, mean(skipmissing(train.visibilite_moy)))
test_full.vitesse_vent_moy = coalesce.(test_full.vitesse_vent_moy, mean(skipmissing(train.vitesse_vent_moy)))

println("Test data prepared. Shape: ", size(test_full))

In [ ]:
# Entraîner sur TOUTES les données d'entraînement
train_final = dropmissing(train[:, [:PM, :NO2, :visibilite_moy, :vitesse_vent_moy]])

X_final = Matrix(hcat(ones(nrow(train_final)),
                       train_final.NO2,
                       train_final.visibilite_moy,
                       train_final.vitesse_vent_moy))
y_final_log = log.(train_final.PM)

# Entraîner OLS et Ridge
β_ols_final = ridge_regression(X_final, y_final_log, 0.0)
β_ridge_final = ridge_regression(X_final, y_final_log, λ_optimal)

println("\nCoefficients OLS:")
println("  β0 (intercept): ", round(β_ols_final[1], digits=4))
println("  β_NO2: ", round(β_ols_final[2], digits=4))
println("  β_visibilite: ", round(β_ols_final[3], digits=4))
println("  β_vitesse_vent: ", round(β_ols_final[4], digits=4))

println("\nCoefficients Ridge (λ=$λ_optimal):")
println("  β0 (intercept): ", round(β_ridge_final[1], digits=4))
println("  β_NO2: ", round(β_ridge_final[2], digits=4))
println("  β_visibilite: ", round(β_ridge_final[3], digits=4))
println("  β_vitesse_vent: ", round(β_ridge_final[4], digits=4))

println("\nNote: Les coefficients Ridge sont légèrement plus petits (rétrécissement vers zéro)")

In [ ]:
# Construire la matrice test
test_clean = dropmissing(test_full[:, [:PM, :NO2, :visibilite_moy, :vitesse_vent_moy, :Date, :stationId]])

X_test = Matrix(hcat(ones(nrow(test_clean)),
                      test_clean.NO2,
                      test_clean.visibilite_moy,
                      test_clean.vitesse_vent_moy))

# Prédictions OLS
y_test_pred_ols_log = X_test * β_ols_final
y_test_pred_ols = exp.(y_test_pred_ols_log)

# Prédictions Ridge
y_test_pred_ridge_log = X_test * β_ridge_final
y_test_pred_ridge = exp.(y_test_pred_ridge_log)

println("\nPrédictions générées.")
println("Nombre de prédictions OLS: ", length(y_test_pred_ols))
println("Nombre de prédictions Ridge: ", length(y_test_pred_ridge))
println("\nPremières prédictions OLS:", y_test_pred_ols[1:5])
println("Premières prédictions Ridge:", y_test_pred_ridge[1:5])

In [ ]:
# Créer les fichiers de soumission
function createID(date::Vector, station::Vector)
    return [(d, s) for (d, s) in zip(date, station)]
end

# Pour OLS
submission_ols = DataFrame(
    ID = createID(test_clean.Date, test_clean.stationId),
    PM = y_test_pred_ols
)

# Pour Ridge
submission_ridge = DataFrame(
    ID = createID(test_clean.Date, test_clean.stationId),
    PM = y_test_pred_ridge
)

CSV.write("predictions_ols.csv", submission_ols)
CSV.write("predictions_ridge_lambda_optimal.csv", submission_ridge)

println("\nFichiers de soumission créés:")
println("  - predictions_ols.csv")
println("  - predictions_ridge_lambda_optimal.csv")

---

## 5. Conclusion et recommandations

### Récapitulatif des améliorations

1. **Analyse de multicollinéarité** :
   - Identifié les corrélations entre prédicteurs
   - Justifié l'utilisation de Ridge regression

2. **Ridge Regression** :
   - Implémentation analytique de Ridge par validation temporelle
   - Sélection automatique de λ optimal
   - Comparaison quantitative OLS vs Ridge

3. **Validation temporelle** :
   - Split 80/20 chronologique pour simuler Kaggle
   - Diagnostic des résidus
   - Estimation réaliste de la généralisation

### Points clés pour l'amélioration future

1. **Ingénierie des variables** :
   - Créer des interactions (ex: NO2 × vitesse_vent)
   - Variables décalées (effet du jour précédent)
   - Variables cycliques (jour de l'année, jour de la semaine)

2. **Autres méthodes de régularisation** :
   - Lasso (sélection de variables automatique)
   - ElasticNet (combinaison Ridge + Lasso)
   - Validation croisée k-fold pour plus de stabilité

3. **Modèles avancés** :
   - Forêts aléatoires ou gradient boosting (captures relations non-linéaires)
   - Modèles de séries temporelles (ARIMA, Prophet)
   - Réseaux de neurones (avec régularisation L1/L2)

4. **Gestion des données** :
   - Étudier les valeurs aberrantes vs réelles
   - Imputation sophistiquée plutôt que moyenne simple
   - Analyse par station (certaines peuvent être "différentes")

### Fichiers générés

- `predictions_ols.csv` : Prédictions OLS simple (baseline)
- `predictions_ridge_lambda_optimal.csv` : Prédictions Ridge régularisées

**Conseil** : Tester les deux sur Kaggle et garder celui avec le meilleur score !